# First test of the modeling part of the pipeline: responsible for creating the model

## Notebook structure (recommended)
2. Data loading: load preprocessed datasets and metadata.
4. Model: define, train, persist.
5. Evaluation: compute and save metrics and plots.
6. Save artifacts: model, transformers, metrics, config.

In [13]:
import pandas as pd
import numpy as np

df_train = pd.read_csv('../data/churn/train.csv')

In [14]:
df_train.head()

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,0,15674932,Okwudilichukwu,668,France,Male,33.0,3,0.00,2,1.0,0.0,181449.97,0
1,1,15749177,Okwudiliolisa,627,France,Male,33.0,1,0.00,2,1.0,1.0,49503.50,0
2,2,15694510,Hsueh,678,France,Male,40.0,10,0.00,2,1.0,0.0,184866.69,0
3,3,15741417,Kao,581,France,Male,34.0,2,148882.54,1,1.0,1.0,84560.88,0
4,4,15766172,Chiemenam,716,Spain,Male,33.0,5,0.00,2,1.0,1.0,15068.83,0


In [ ]:
# Define the specific dates range
start_date = "2023-06-01"
end_date = "2023-09-30"

safra = []
# Generate a random date within the specified range
for i in range(df_train.shape[0]):
    safra.append(
        pd.to_datetime(
            np.random.choice(pd.date_range(start=start_date, end=end_date))
        ).strftime("%Y%m")
    )

df_train["safra"] = safra

rng = np.random.RandomState(42)
n_ones = int(round(0.2 * len(df_train)))

df_train['no_action'] = 0
df_train.loc[rng.choice(df_train.index, size=n_ones, replace=False), 'no_action'] = 1

In [19]:
df_train['no_action'].value_counts(normalize=True)

no_action
0    0.799999
1    0.200001
Name: proportion, dtype: float64

In [20]:
df_train['safra'].value_counts(normalize=True)

safra
202307    0.253651
202308    0.253633
202306    0.246555
202309    0.246161
Name: proportion, dtype: float64

In [21]:
df_oot = df_train[df_train['safra'] == '202309']
df_train = df_train[df_train['safra'] != '202309']

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
import lightgbm as lgb

# Train a LightGBM classifier for churn prediction (new notebook cell)


# feature setup: drop identifiers and text surname
drop_cols = ['id', 'CustomerId', 'Surname', 'safra', 'no_action']
target = 'Exited'

X = df_train.drop(columns=drop_cols + [target])
y = df_train[target]

# mark categorical features so LightGBM can handle them natively
for c in ['Geography', 'Gender']:
    if c in X.columns:
        X[c] = X[c].astype('category')
    if c in df_train.columns:
        df_train[c] = df_train[c].astype('category')

# train/validation split
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# model
model = lgb.LGBMClassifier(random_state=42)

# fit with early stopping
model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    categorical_feature=['Geography', 'Gender']
)

# validation metrics
val_probs = model.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)
print("Validation AUC:", round(roc_auc_score(y_val, val_probs), 4))
print("Validation Accuracy:", round(accuracy_score(y_val, val_preds), 4))
print(classification_report(y_val, val_preds, digits=4))

[LightGBM] [Info] Number of positive: 21042, number of negative: 78485
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000826 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 860
[LightGBM] [Info] Number of data points in the train set: 99527, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.211420 -> initscore=-1.316387
[LightGBM] [Info] Start training from score -1.316387
Validation AUC: 0.8882
Validation Accuracy: 0.8643
              precision    recall  f1-score   support

           0     0.8881    0.9473    0.9167     19621
           1     0.7384    0.5546    0.6335      5261

    accuracy                         0.8643     24882
   macro avg     0.8132    0.7510    0.7751     24882
weighted avg     0.8564    0.8643    0.8568     24882

